In [9]:
import os
import librosa
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (Wav2Vec2Processor, Wav2Vec2ForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)

# Define dataset directory
dataset_dir = "/content/drive/My Drive/archive"  # Adjust if needed
metadata_path = os.path.join(dataset_dir, "UrbanSound8K.csv")
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = [os.path.join(dataset_dir, "fold{}".format(row['fold']), row["slice_file_name"]) for _, row in metadata.iterrows()]
labels = metadata["classID"].values

# Train-test split
from sklearn.model_selection import train_test_split
train_paths, test_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)

# Load pre-trained processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

# Function to preprocess audio
def preprocess_audio(file_path):
    try:
        # Load audio file
        audio, sr = librosa.load(file_path, sr=16000)  # Resample to 16kHz

        # Process audio using Wav2Vec2 processor with max_length
        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding="max_length",  # Ensures all inputs are the same length
            max_length=16000 * 5,  # 5 seconds of audio
            truncation=True  # Ensures audio longer than max_length is truncated
        )

        return inputs.input_values.squeeze(0)  # Remove extra dimension

    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


# Custom Dataset class
class UrbanSoundDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

        # Keep only valid file paths
        self.valid_data = [(fp, lbl) for fp, lbl in zip(file_paths, labels) if os.path.exists(fp)]

    def __len__(self):
        return len(self.valid_data)

    def __getitem__(self, idx):
        file_path, label = self.valid_data[idx]
        input_values = preprocess_audio(file_path)

        # Handle cases where audio processing fails
        if input_values is None:
            input_values = torch.zeros((16000,))  # 1-second silence as a placeholder

        return {"input_values": input_values, "labels": torch.tensor(label, dtype=torch.long)}

# Create Datasets
train_dataset = UrbanSoundDataset(train_paths, y_train)
test_dataset = UrbanSoundDataset(test_paths, y_test)

# Load pre-trained model
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base", num_labels=10
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

# Define Trainer arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none"
)

# Use DataCollatorWithPadding to handle dynamic input lengths
data_collator = DataCollatorWithPadding(tokenizer=processor.feature_extractor)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator  # Added data collator
)

# Train the model
trainer.train()

# Evaluate the model
metrics = trainer.evaluate()
print("Evaluation Results:", metrics)


/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:312: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,1.387100,1.246397
2,0.869300,0.766931
3,0.532500,0.488326
4,0.317100,0.451148
5,0.325800,0.498630
6,0.240200,0.527587
7,0.357000,0.379652
8,0.095700,0.369819
9,0.004700,0.299712


Epoch,Training Loss,Validation Loss
1,1.387100,1.246397
2,0.869300,0.766931
3,0.532500,0.488326
4,0.317100,0.451148
5,0.325800,0.498630
6,0.240200,0.527587
7,0.357000,0.379652
8,0.095700,0.369819
9,0.004700,0.299712
10,0.141800,0.359724


Evaluation Results: {'eval_loss': 0.35972392559051514, 'eval_runtime': 38.664, 'eval_samples_per_second': 30.778, 'eval_steps_per_second': 3.854, 'epoch': 10.0}


In [10]:
import numpy as np

# Get predictions
predictions = trainer.predict(test_dataset)

# Extract logits and labels
logits = predictions.predictions
labels = predictions.label_ids

# Convert logits to class predictions
preds = np.argmax(logits, axis=-1)

# Compute accuracy
accuracy = np.mean(preds == labels)

print(f"Model Accuracy: {accuracy * 100:.2f}%")


Model Accuracy: 94.03%
